[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/petro1eum/trust_chain/blob/main/examples/trustchain_full_demo.ipynb)

# TrustChain Full Demo with Optional Real LLM

This notebook demonstrates selected TrustChain capabilities with a deterministic offline workflow. A live OpenRouter call requires explicit `RUN_OPENROUTER = True` plus externally configured `OPENROUTER_API_KEY` and `OPENROUTER_MODEL`.

**Features tested:**
1. Basic signing & verification
2. Chain of Trust (linked operations)
3. Multi-tool agent workflow
4. Merkle audit trail
5. Tamper detection
6. Audit export (JSON)
7. Multi-tenant isolation

Reviewed with TrustChain 3.3 / Python 3.12. This notebook uses synthetic data. Offline examples use an in-memory log; production persistence, PKI and authorization require configuration. Signatures establish attribution and byte integrity, not the truth of a tool result. [Notebook guide](https://github.com/petro1eum/trust_chain/blob/main/examples/README.md).

In [ ]:
%pip install -q --upgrade 'trustchain[ai]>=3.3.0,<4'

In [ ]:
# Explicit opt-in; an ambient credential never triggers a paid call.
import os
RUN_OPENROUTER = False
API_KEY = os.environ.get("OPENROUTER_API_KEY") if RUN_OPENROUTER else None
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL")
if RUN_OPENROUTER and (not API_KEY or not OPENROUTER_MODEL):
    raise RuntimeError("Configure provider credentials/model outside the notebook before opting in")
print("Execution mode:", "OpenRouter" if API_KEY else "offline fixture; no model call")


In [ ]:
# Setup
import copy
import json
import tempfile
from types import SimpleNamespace

from trustchain import TrustChain, TrustChainConfig
from trustchain.v2.merkle import MerkleTree, verify_proof
from trustchain.v2.tenants import TenantManager

# OpenRouter client is optional. Without a key, use a tiny deterministic
# stand-in so Run all works in a fresh Colab runtime.
client = None
if API_KEY:
    from openai import OpenAI
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=API_KEY,
    )

# TrustChain instance
tc = TrustChain(TrustChainConfig(enable_nonce=False))

print('✅ TrustChain ready')
print('🌐 OpenRouter client:', 'enabled' if client else 'offline demo mode')
print(f'🔑 Key ID: {tc.get_key_id()}')

In [ ]:
import ast
from decimal import Decimal, localcontext

def calculate(expression):
    if not isinstance(expression, str) or len(expression) > 256:
        raise ValueError("Expression too large")
    tree = ast.parse(expression, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > 64:
        raise ValueError("Too many operations")
    def visit(node, depth=0):
        if depth > 16:
            raise ValueError("Expression too deep")
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            value = Decimal(ast.get_source_segment(expression, node))
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = visit(node.operand, depth+1)
            if isinstance(node.op, ast.USub): value = -value
        elif isinstance(node, ast.BinOp):
            a, b = visit(node.left, depth+1), visit(node.right, depth+1)
            if isinstance(node.op, ast.Add): value = a+b
            elif isinstance(node.op, ast.Sub): value = a-b
            elif isinstance(node.op, ast.Mult): value = a*b
            elif isinstance(node.op, ast.Div): value = a/b
            elif isinstance(node.op, ast.Pow) and b == int(b) and 0 <= b <= 12: value = a**int(b)
            else: raise ValueError("Operator forbidden")
        else:
            raise ValueError("Only bounded arithmetic is supported")
        if not value.is_finite() or abs(value) > Decimal("1e12"):
            raise ValueError("Result out of range")
        return value
    with localcontext() as ctx:
        ctx.prec = 30
        return str(visit(tree.body))
assert calculate("2 ** 10") == "1024"
try:
    calculate("__import__('os').getcwd()")
except ValueError:
    pass
else:
    raise AssertionError("Arbitrary code must be rejected")

---
## 1️⃣ Basic Signing & Verification

In [ ]:
# Sign a tool fact using the public TrustChain API — auto-chain + auto-commit.
signed = tc.sign(
    tool_id='test_tool',
    data={'message': 'Hello TrustChain!'},
)

print('📝 Signed response:')
print(f'   Tool: {signed.tool_id}')
print(f'   Data: {signed.data}')
print(f'   Signature: {signed.signature[:40]}...')
print(f'   Timestamp: {signed.timestamp}')
print(f'\n✅ Verified: {tc.verify(signed)}')

---
## 2️⃣ Chain of Trust with an optional model call

In [ ]:
# Define tools for the LLM
tools = [
    {'type': 'function', 'function': {'name': 'get_weather', 'parameters': {'type': 'object', 'properties': {'city': {'type': 'string'}}}}},
    {'type': 'function', 'function': {'name': 'calculate', 'parameters': {'type': 'object', 'properties': {'expr': {'type': 'string'}}}}},
    {'type': 'function', 'function': {'name': 'search', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}}}}
]

# Tool implementations
def execute_tool(name, args):
    if name == 'get_weather':
        return f"Sunny, 22°C in {args.get('city', 'Unknown')}"
    elif name == 'calculate':
        return calculate(args.get('expr', '0'))
    elif name == 'search':
        return f"Found 5 results for: {args.get('query', '')}"
    return 'Unknown'


def offline_tool_call(name, args):
    return SimpleNamespace(
        function=SimpleNamespace(
            name=name,
            arguments=json.dumps(args),
        )
    )


# Multi-step agent workflow
chain = []
queries = [
    'What is the weather in Tokyo? Use the tool.',
    'Calculate 2 ** 10 using the calculate tool.',
    'Search for AI security using the search tool.'
]
offline_calls = [
    [offline_tool_call('get_weather', {'city': 'Tokyo'})],
    [offline_tool_call('calculate', {'expr': '2 ** 10'})],
    [offline_tool_call('search', {'query': 'AI security'})],
]

print('🤖 Running multi-step agent workflow...\n')

for i, query in enumerate(queries, 1):
    print(f'📤 Step {i}: {query}')

    if client:
        response = client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[{'role': 'user', 'content': query}],
            tools=tools,
            tool_choice='auto',
            max_tokens=100
        )
        tool_calls = response.choices[0].message.tool_calls or []
    else:
        tool_calls = offline_calls[i - 1]

    for tc_call in tool_calls:
        args = json.loads(tc_call.function.arguments)
        result = execute_tool(tc_call.function.name, args)

        # tc.sign() auto-chains from chain HEAD — no manual parent threading.
        # Pass parent_signature=None only if you want to start a NEW root.
        signed = tc.sign(
            tool_id=tc_call.function.name,
            data={'args': args, 'result': result},
        )
        parent_sig = signed.parent_signature
        chain.append(signed)

        print(f'   ✅ {tc_call.function.name}: {result}')
        print(f'   🔏 Signature: {signed.signature[:32]}...')
        if parent_sig:
            print(f'   ⛓️ Linked to parent: {parent_sig[:32]}...')
    print()

print(f'📊 Chain complete: {len(chain)} tool calls')
print(f'✅ All signatures valid: {all(tc.verify(s) for s in chain)}')

assert all(tc.verify(s) for s in chain)
if client is None:
    assert [s.tool_id for s in chain] == ['get_weather', 'calculate', 'search']
    assert chain[1].data['result'] == '1024'
    assert all(chain[i].parent_signature == chain[i-1].signature for i in range(1, len(chain)))


---
## 3️⃣ Merkle Audit Trail

In [ ]:
# Build Merkle tree from chain signatures
signatures = [s.signature for s in chain]
tree = MerkleTree.from_chunks(signatures)

print('🌳 Merkle Tree Audit Trail')
print(f'   Root: {tree.root[:48]}...')
print(f'   Leaves: {len(tree.leaves)}')

# Verify inclusion proof
proof = tree.get_proof(0)
valid = verify_proof(signatures[0], proof, tree.root)
print('\n🔍 Verifying signature[0] is in tree...')
print(f'   Proof length: {len(proof.proof) if hasattr(proof, "proof") else "N/A"}')
print(f'   ✅ Verified: {valid}')

assert valid


---
## 4️⃣ Tamper Detection

In [ ]:
# Copy original and tamper with it
original = chain[0]
tampered = copy.deepcopy(original)
tampered.data['result'] = 'HACKED: 999999 BTC'

print('🛡️ Tamper Detection Test')
print(f'\n   Original data: {original.data}')
print(f'   Tampered data: {tampered.data}')
print(f'\n   Original verified: {tc.verify(original)} ✅')
print(f'   Tampered verified: {tc.verify(tampered)} ❌')

assert tc.verify(original)
assert not tc.verify(tampered)


---
## 5️⃣ Multi-Tenant Isolation

In [ ]:
with tempfile.TemporaryDirectory() as key_dir:
    # Create tenant manager
    manager = TenantManager(key_storage_dir=key_dir)

    # Create separate tenants
    tenants = {
        name: manager.get_or_create(name)
        for name in ['company_a', 'company_b', 'company_c']
    }

    print('🏢 Multi-Tenant Isolation Test')
    for name, tenant_tc in tenants.items():
        print(f'   {name}: Key {tenant_tc.get_key_id()[:20]}...')

    # Sign data with tenant A (public API — keys stay strictly per-tenant)
    data_a = tenants['company_a'].sign('order', {'id': 12345, 'amount': 100})
    print('\n📝 Company A signed order #12345')

    # Try to verify with tenant B (should fail)
    can_b_verify_a = tenants['company_b'].verify(data_a)
    print(f'   Company B can verify A\'s data: {can_b_verify_a} ❌')

    # Original tenant can verify (should pass)
    can_a_verify_a = tenants['company_a'].verify(data_a)
    print(f'   Company A can verify own data: {can_a_verify_a} ✅')

assert not can_b_verify_a and can_a_verify_a


---
## 6️⃣ Export Audit Trail

In [ ]:
# Audit trail export — OSS path.
# ChainExplorer HTML export requires the authorized Pro wheel or private checkout.
audit_rows = [
    {
        'tool_id': r.tool_id,
        'signature': r.signature,
        'data': r.data,
        'timestamp': getattr(r, 'response_timestamp', None),
    }
    for r in chain
]
json_export = json.dumps(audit_rows, indent=2, default=str)
parsed = json.loads(json_export)

print('\U0001f4e4 Audit Trail Export (OSS)')
print(f'   Entries: {len(parsed)}')
print('\n\U0001f4c4 JSON Preview:')
for entry in parsed:
    print(f"   - {entry['tool_id']}: sig={entry['signature'][:20]}...")

print('\n\U0001f4a1 Pro tier adds HTML/PDF ComplianceReport and ChainExplorer:')
print('   Install the authorized Pro wheel or private checkout; see the Pro guide.')
print('   from trustchain_pro import ChainExplorer, ComplianceReport')

assert len(parsed) == len(chain)


## 🧾 Export a `.tcreceipt` — portable, offline-verifiable proof

All the signing above lives inside your TrustChain instance. Sometimes you need to
hand someone *else* — a regulator, a partner, an end user — a self-contained
artifact they can verify on their own laptop, without your server.

That's `.tcreceipt`: one JSON file, one drag-and-drop into
[`examples/verify.html`](https://github.com/petro1eum/trust_chain/blob/main/examples/verify.html),
WebCrypto Ed25519 check, **no network required**.

In [ ]:
from pathlib import Path

from trustchain import Receipt, build_receipt

# Take any signed response we produced above — re-use the first one.
# In a real agent, you'd loop over the facts you want to expose.
receipt = build_receipt(
    signed,
    tc.export_public_key(),
    key_id=tc.get_key_id(),
)

out = Path("demo_fact.tcreceipt")
out.write_text(receipt.to_json(indent=2))
print(f"💾 wrote {out} — {out.stat().st_size} bytes")

# Roundtrip: load + verify + pin to a known public key
loaded = Receipt.load(out.read_text())
v = loaded.verify(
    expected_public_key_b64=tc.export_public_key(),
    max_age_seconds=3600,
)
print(f"\n✅ valid       : {v.valid}")
print(f"   sig_ok     : {v.signature_ok}")
print(f"   identity_ok: {v.identity_ok}")
print(f"   errors     : {v.errors or '—'}")
print(f"   warnings   : {v.warnings or '—'}")

# Try it in the browser! Download examples/verify.html from the repo,
# open it locally (file://), drag demo_fact.tcreceipt onto the drop zone.
# You'll get the SAME verdict, no Python, no server.

assert v.valid and v.signature_ok


---
## ✅ Summary

The assertions above check the offline examples in this notebook. Live provider execution requires explicit opt-in and is not implied by this summary:

| Feature | Status |
|---------|--------|
| Basic signing | ✅ |
| Verification | ✅ |
| Chain of Trust | ✅ |
| Optional real LLM tool calls | Not run by default |
| Merkle audit trail | ✅ |
| Tamper detection | ✅ |
| Multi-tenant isolation | ✅ |
| Audit export | JSON checked; HTML requires Pro |